# Contagion researcher workbench

Optional dependencies: `python -m pip install -e ".[analysis]"`. No inference or network calls. Select one frozen cohort before plotting effects. Raw logs remain authoritative; these are validated public projections.


In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from contagion.research_publication import validate_research_snapshot
from contagion.evidence_stats import summaries, paired_effects, neural_diagnostics
from contagion.reporting import generate_report
repo = Path.cwd() if Path('dashboard').exists() else Path.cwd().parent
snapshot = repo / 'dashboard/data/research.json'
parsed = validate_research_snapshot(snapshot)
rows = [r.model_dump() for r in parsed.studies]
runs = pd.DataFrame(rows)
runs.groupby(['backend','model','protocol','split','task_family','status'], dropna=False).size().rename('attempts')


## Choose a cohort explicitly

A different freeze may represent a prompt, schema, task, seed schedule or source change. Do not pool those revisions. Replace `None` with a selected freeze hash after inspecting this inventory. No automatic best-result selection.


In [ ]:
display(runs[['model','worker_label','protocol','task_family','split','freeze_hash']].drop_duplicates())
FREEZE_HASH = None
cohort = runs[runs['freeze_hash'].eq(FREEZE_HASH)].copy()
if cohort.empty:
    print('Select a freeze before drawing cohort-specific conclusions.')
else:
    display(pd.DataFrame(summaries(cohort.to_dict('records'))))
    display(pd.DataFrame(paired_effects(cohort.to_dict('records'))))


In [ ]:
if not cohort.empty:
    sns.catplot(data=cohort, x='topology', y='task_success_rate', hue='intervention', col='agent_count', kind='strip', jitter=.08)
    plt.show()


## Monitoring and neural quality

Visibility projections are dependent and recall is undefined without positives. Random-weight captures validate instrumentation only. Selected-neuron distributions are biased samples, not all-neuron or SAE distributions.


In [ ]:
monitor = pd.DataFrame([r.model_dump() for r in parsed.monitor])
if not monitor.empty:
    display(monitor[['model','protocol','cohort','topology','policy','visibility','n','tp','fp','tn','fn','recall','false_positive_rate']])
display(pd.DataFrame([neural_diagnostics(r.model_dump()) for r in parsed.neural]))


In [ ]:
generate_report(snapshot, repo / 'results/paper-draft', legacy_path=repo / 'dashboard/data/results.json')
